In [6]:
import pandas as pd
import numpy as np
import logging
from typing import Dict, List, Tuple, Optional, Union
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from OprFuncs import *
#from langchain.schema.runnable import RunnableSequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.agents import AgentExecutor, Tool, create_react_agent
#from langchain import hub
import re
#from modelEXT.PygalCodeComponents import PygalCodeComponents
#from langchain.output_parsers import PydanticOutputParser
from DatabaseManager import DatabaseManager
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.figure import Figure
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import datetime

# Setup logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

class DataAnalyzer:
    """
    A class for data analysis and cleaning using Large Language Models (LLMs).
    
    This class provides a comprehensive interface for data cleaning, analysis,
    generating recommendations, and creating visualizations using large language models.
    
    Attributes:
        dataframe (pd.DataFrame): The current dataframe being analyzed
        original_dataframe (pd.DataFrame): A copy of the original dataframe before cleaning
        llm: The large language model used for analysis
        data_info (str): Automatically inferred information about the data
        data_description (str): Description of the data
        data_sample (str): Sample of the data (first rows)
        data_cols (str): List of column names separated by commas
        cleaning_log (list): Log of cleaning operations performed
        
    Examples:
        >>> from DataAnalyzer import DataAnalyzer
        >>> import pandas as pd
        >>> from langchain_openai import ChatOpenAI
        >>> 
        >>> # Create a new instance
        >>> df = pd.read_csv("data.csv")
        >>> llm = ChatOpenAI()
        >>> analyzer = DataAnalyzer(df, llm)
        >>> 
        >>> # Clean the data
        >>> strategies = analyzer.recommend_cleaning_strategy()
        >>> cleaned_df = analyzer.clean_data(strategies)
        >>> 
        >>> # Analyze the data
        >>> analysis = analyzer.analysis_data()
        >>> 
        >>> # Generate questions and recommendations
        >>> questions = analyzer.questions_gen(5)
        >>> recommendations = analyzer.generate_recommendations()
    """
    def __init__(self,dataframe,llm,user_id=None):
        """
        Initialize the data analyzer with a dataframe and a large language model.
        
        Parameters:
            dataframe (pd.DataFrame): The dataframe to analyze
            llm: The large language model to use for analysis
            user_id (str, optional): User ID for tracking analytics. Default is None.
            
        Example:
            >>> analyzer = DataAnalyzer(df, llm, user_id="user123")
        """
        self.dataframe = dataframe
        self.original_dataframe = dataframe.copy()  # Store original for restore capability
        self.llm = llm
        self.data_info = data_infer(dataframe)
        self.data_description = data_describer(dataframe)
        self.data_sample = dataframe.head().to_string()
        self.data_cols = ", ".join(dataframe.columns)
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        self.user_id = user_id
        self.cleaning_log = []  # Log of cleaning operations

    def recommend_cleaning_strategy(self) -> Dict:
        """
        Analyze the dataframe and recommend data cleaning strategies.
        
        This function examines the data and identifies:
        - Missing values and the best method to handle them
        - Duplicate values
        - Outliers in numerical columns
        - Special characters in text columns
        - Potential data type conversions
        
        Returns:
            Dict: Dictionary containing recommended cleaning strategies for each type of data issue
        
        Strategies include:
            - missing_values: Strategies for handling missing values for each column
            - duplicates: Strategy for handling duplicate rows
            - outliers: Strategies for handling outliers for each numerical column
            - special_chars: Strategy for handling special characters
            - data_types: Suggested data type conversions
            
        Example:
            >>> strategies = analyzer.recommend_cleaning_strategy()
            >>> print(strategies)
            {
                'missing_values': {'age': 'median', 'city': 'mode'},
                'duplicates': 'drop_first',
                'outliers': {'salary': 'iqr', 'age': 'iqr'},
                'special_chars': {'strategy': 'remove', 'columns': ['name', 'address']},
                'data_types': {'date_column': 'datetime'}
            }
        """
        try:
            df = self.dataframe
            recommendations = {}
            
            # Check for missing values
            missing_counts = df.isna().sum()
            missing_cols = missing_counts[missing_counts > 0]
            
            if len(missing_cols) > 0:
                # Recommend strategy based on column type and missing percentage
                missing_strategy = {}
                for col in missing_cols.index:
                    missing_pct = missing_counts[col] / len(df)
                    
                    if missing_pct > 0.5:
                        # Too many missing values, consider dropping the column
                        missing_strategy[col] = "drop_column"
                    elif df[col].dtype in [np.float64, np.int64]:
                        # For numeric columns, use median
                        missing_strategy[col] = "median"
                    else:
                        # For categorical columns, use mode
                        missing_strategy[col] = "mode"
                
                recommendations['missing_values'] = missing_strategy
            
            # Check for duplicates
            duplicate_count = df.duplicated().sum()
            if duplicate_count > 0:
                recommendations['duplicates'] = 'drop_first'
            
            # Check for outliers in numeric columns
            numeric_cols = df.select_dtypes(include=np.number).columns
            outlier_strategy = {}
            
            for col in numeric_cols:
                # Use IQR to detect outliers
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                
                outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                
                if outlier_count > 0 and outlier_count / len(df) < 0.05:
                    # Small percentage of outliers, use IQR clipping
                    outlier_strategy[col] = "iqr"
            
            if outlier_strategy:
                recommendations['outliers'] = outlier_strategy
            
            # Check for special characters in string columns
            string_cols = df.select_dtypes(include=['object']).columns
            special_chars_cols = []
            
            for col in string_cols:
                if df[col].dtype == 'object':
                    # Check for special characters
                    has_special = df[col].astype(str).str.contains(r'[^\w\s]', regex=True).any()
                    if has_special:
                        special_chars_cols.append(col)
            
            if special_chars_cols:
                recommendations['special_chars'] = {
                    'strategy': 'remove',
                    'columns': special_chars_cols
                }
            
            # Check for potential data type conversions
            type_conversions = {}
            
            # Check for datetime columns
            for col in df.columns:
                if df[col].dtype == 'object':
                    # Try to convert to datetime
                    try:
                        pd.to_datetime(df[col], errors='raise')
                        type_conversions[col] = 'datetime'
                    except:
                        pass
            
            if type_conversions:
                recommendations['data_types'] = type_conversions
            
            return recommendations
            
        except Exception as e:
            logger.error(f"Error recommending cleaning strategy: {str(e)}")
            return {}

    def clean_data(self, strategies: Dict = None) -> pd.DataFrame:
        """
        Clean the dataset using various strategies.
        
        Parameters:
            strategies (Dict, optional): Dictionary of cleaning strategies to apply.
                Available strategies:
                - missing_values: 'drop', 'mean', 'median', 'mode', 'zero', or dict of custom values per column
                - duplicates: 'drop_first', 'drop_last', 'keep'
                - outliers: 'clip', 'remove', 'iqr', 'zscore'
                - special_chars: 'remove', 'replace'
                - data_types: Dict mapping column names to desired data types
                
        Returns:
            pd.DataFrame: Cleaned dataframe
            
        Notes:
            - If strategies is None, default strategies will be used.
            - All cleaning operations are logged in self.cleaning_log.
            - Derived properties like data_info and data_description are updated after cleaning.
            
        Examples:
            # Using recommended strategies
            >>> recommended = analyzer.recommend_cleaning_strategy()
            >>> cleaned_df = analyzer.clean_data(recommended)
            
            # Specifying custom strategies
            >>> custom_strategies = {
            ...     'missing_values': 'mean',
            ...     'duplicates': 'drop_first',
            ...     'outliers': 'zscore'
            ... }
            >>> cleaned_df = analyzer.clean_data(custom_strategies)
            
            # Specifying column-specific strategies
            >>> column_specific = {
            ...     'missing_values': {'age': 0, 'income': 'median', 'city': 'mode'},
            ...     'outliers': 'iqr'
            ... }
            >>> cleaned_df = analyzer.clean_data(column_specific)
        """
        try:
            # Start with a fresh copy of the original data
            df = self.original_dataframe.copy()
            
            # Default strategies if none provided
            if strategies is None:
                strategies = {
                    'missing_values': 'median',
                    'duplicates': 'drop_first',
                    'outliers': 'iqr',
                    'special_chars': 'remove'
                }
            
            logger.info("Starting data cleaning process")
            self.cleaning_log = []  # Reset cleaning log
            
            # 1. Handle missing values
            if 'missing_values' in strategies:
                strategy = strategies['missing_values']
                missing_count_before = df.isna().sum().sum()
                
                if strategy == 'drop':
                    df = df.dropna()
                    self.cleaning_log.append(f"Dropped {missing_count_before} missing values")
                
                elif strategy in ['mean', 'median', 'mode']:
                    for col in df.select_dtypes(include=np.number).columns:
                        if df[col].isna().sum() > 0:
                            if strategy == 'mean':
                                df[col] = df[col].fillna(df[col].mean())
                            elif strategy == 'median':
                                df[col] = df[col].fillna(df[col].median())
                            elif strategy == 'mode':
                                df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 0)
                    
                    # For non-numeric columns, use mode
                    for col in df.select_dtypes(exclude=np.number).columns:
                        if df[col].isna().sum() > 0:
                            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "")
                            
                    self.cleaning_log.append(f"Filled {missing_count_before} missing values using {strategy}")
                
                elif strategy == 'zero':
                    df = df.fillna(0)
                    self.cleaning_log.append(f"Filled {missing_count_before} missing values with zero")
                
                elif isinstance(strategy, dict):
                    # Custom value for each column
                    for col, value in strategy.items():
                        if col in df.columns:
                            df[col] = df[col].fillna(value)
                    self.cleaning_log.append(f"Filled missing values with custom values for specified columns")
            
            # 2. Handle duplicates
            if 'duplicates' in strategies:
                strategy = strategies['duplicates']
                duplicate_count = df.duplicated().sum()
                
                if strategy == 'drop_first':
                    df = df.drop_duplicates(keep='first')
                    self.cleaning_log.append(f"Removed {duplicate_count} duplicate rows (keeping first occurrence)")
                
                elif strategy == 'drop_last':
                    df = df.drop_duplicates(keep='last')
                    self.cleaning_log.append(f"Removed {duplicate_count} duplicate rows (keeping last occurrence)")
            
            # 3. Handle outliers
            if 'outliers' in strategies:
                strategy = strategies['outliers']
                numeric_cols = df.select_dtypes(include=np.number).columns
                
                if strategy == 'iqr':
                    # IQR method
                    for col in numeric_cols:
                        Q1 = df[col].quantile(0.25)
                        Q3 = df[col].quantile(0.75)
                        IQR = Q3 - Q1
                        lower_bound = Q1 - 1.5 * IQR
                        upper_bound = Q3 + 1.5 * IQR
                        
                        outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                        if outliers_count > 0:
                            df.loc[df[col] < lower_bound, col] = lower_bound
                            df.loc[df[col] > upper_bound, col] = upper_bound
                            self.cleaning_log.append(f"Clipped {outliers_count} outliers in column '{col}' using IQR method")
                
                elif strategy == 'zscore':
                    # Z-score method
                    from scipy import stats
                    for col in numeric_cols:
                        z_scores = np.abs(stats.zscore(df[col].fillna(df[col].median())))
                        outliers = z_scores > 3
                        outliers_count = outliers.sum()
                        
                        if outliers_count > 0:
                            df.loc[outliers, col] = df[col].median()
                            self.cleaning_log.append(f"Replaced {outliers_count} outliers in column '{col}' using Z-score method")
                
                elif strategy == 'remove':
                    for col in numeric_cols:
                        Q1 = df[col].quantile(0.25)
                        Q3 = df[col].quantile(0.75)
                        IQR = Q3 - Q1
                        lower_bound = Q1 - 1.5 * IQR
                        upper_bound = Q3 + 1.5 * IQR
                        
                        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
                        outliers_count = outlier_mask.sum()
                        
                        if outliers_count > 0:
                            df = df[~outlier_mask]
                            self.cleaning_log.append(f"Removed {outliers_count} rows with outliers in column '{col}'")
            
            # 4. Handle special characters
            if 'special_chars' in strategies:
                strategy = strategies['special_chars']
                string_cols = df.select_dtypes(include=['object']).columns
                
                if strategy == 'remove':
                    for col in string_cols:
                        if df[col].dtype == 'object':
                            # Replace special characters with empty string
                            df[col] = df[col].astype(str).str.replace(r'[^\w\s]', '', regex=True)
                    self.cleaning_log.append(f"Removed special characters from text columns")
                
                elif strategy == 'replace':
                    for col in string_cols:
                        if df[col].dtype == 'object':
                            # Replace special characters with underscore
                            df[col] = df[col].astype(str).str.replace(r'[^\w\s]', '_', regex=True)
                    self.cleaning_log.append(f"Replaced special characters with underscore in text columns")
            
            # 5. Handle data types
            if 'data_types' in strategies and isinstance(strategies['data_types'], dict):
                type_conversions = strategies['data_types']
                
                for col, dtype in type_conversions.items():
                    if col in df.columns:
                        try:
                            if dtype == 'datetime':
                                df[col] = pd.to_datetime(df[col], errors='coerce')
                            else:
                                df[col] = df[col].astype(dtype)
                            self.cleaning_log.append(f"Converted column '{col}' to {dtype} type")
                        except Exception as e:
                            self.cleaning_log.append(f"Failed to convert column '{col}' to {dtype}: {str(e)}")
            
            # Update the dataframe and derived properties
            rows_diff = len(self.dataframe) - len(df)
            cols_diff = len(self.dataframe.columns) - len(df.columns)
            
            self.dataframe = df
            
            # Update derived properties
            self.data_info = data_infer(df)
            self.data_description = data_describer(df)
            self.data_sample = df.head().to_string()
            self.data_cols = ", ".join(df.columns)
            
            logger.info(f"Data cleaning completed. Rows changed: {rows_diff}, Columns changed: {cols_diff}")
            summary = f"Data cleaning completed. Original shape: {self.original_dataframe.shape}, New shape: {df.shape}"
            self.cleaning_log.append(summary)
            
            return df
            
        except Exception as e:
            error_msg = f"Error cleaning data: {str(e)}"
            logger.error(error_msg)
            self.cleaning_log.append(error_msg)
            return self.dataframe

    def restore_original_data(self) -> pd.DataFrame:
        """
        Restore the dataframe to its original state before cleaning.
        
        This function restores the original data and updates all derived properties.
        Useful when you want to reset the data or try different cleaning strategies.
        
        Returns:
            pd.DataFrame: The original dataframe
            
        Example:
            >>> # After cleaning the data
            >>> cleaned_df = analyzer.clean_data(strategies)
            >>> 
            >>> # Return to the original data
            >>> original_df = analyzer.restore_original_data()
            >>> 
            >>> # Apply different strategies
            >>> different_strategies = {...}
            >>> newly_cleaned_df = analyzer.clean_data(different_strategies)
        """
        try:
            self.dataframe = self.original_dataframe.copy()
            
            # Update derived properties
            self.data_info = data_infer(self.dataframe)
            self.data_description = data_describer(self.dataframe)
            self.data_sample = self.dataframe.head().to_string()
            self.data_cols = ", ".join(self.dataframe.columns)
            
            logger.info("Restored original dataframe")
            return self.dataframe
            
        except Exception as e:
            logger.error(f"Error restoring original data: {str(e)}")
            return None
        
    def analysis_data(self):
        """
        Analyze the data using the large language model to create a comprehensive analytical report.
        
        This function sends data information to the large language model to create
        a deep analysis including executive summary, key patterns, statistical validation,
        risks, growth opportunities, and strategic recommendations.
        
        Returns:
            str: The text analysis report
            
        Notes:
            - The analysis result is stored in self.analysis.
            - The conversation history is stored in self.memory.
            - The analysis is saved to the database if self.report_id is set.
            
        Example:
            >>> analysis_report = analyzer.analysis_data()
            >>> print(analysis_report[:500])  # Display just the beginning as the report may be long
        """
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        analysis_template = '''
        You are a data analyst. You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_description}
        You are a **world-class Senior Data Analyst and Applied Statistician**, with deep expertise in business intelligence, behavioral data, financial analytics, and statistical modeling. I will provide you with a dataset in the form of a DataFrame, CSV, or Excel file.

        🎯 Your task is to perform a **comprehensive, statistically-sound, and executive-ready analysis** tailored for decision-makers, technical stakeholders, and strategic planners.

        ---

        ## 🧾 1. Executive Summary
        - Summarize the most important findings, using clear and impactful language.
        - Highlight how these findings affect the business, strategy, or operations.
        - Include headline numbers (KPIs, revenue impact, user behavior shifts...).

        ---

        ## 📊 2. Key Patterns & Strategic Insights
        - Explore key trends, distributions, and variable relationships.
        - Use metrics such as:
        - **Mean, Median, Std. Dev.**
        - **Correlation Coefficients**
        - **Distribution Skewness/Kurtosis**
        - **R² Score (if regression applies)**

        📌 Visuals may include histograms, bar charts, scatter plots, or heatmaps.

        ---

        ## 📐 3. Statistical Validation & Modeling
        - Apply formal **hypothesis tests** where applicable:
        - t-tests, ANOVA, Chi-square, or Z-tests.
        - Report **p-values** and **statistical significance**.
        - Build simple predictive or explanatory models:
        - Linear/Logistic Regression, Decision Trees...
        - Report key metrics:
        - **R²**, **RMSE**, **AUC**, or **F1-Score** (as appropriate).
        - Provide **Confidence Intervals** for estimates when relevant.

        📈 Clearly indicate statistically significant results and what they mean for the business.

        ---

        ## ⚠️ 4. Risks, Anomalies & Data Limitations
        - Identify:
        - Missing values
        - Outliers
        - Sampling bias or measurement error
        - Explain how each issue might impact model validity or business interpretations.
        - Suggest methods for mitigation (e.g., imputation, resampling, anomaly filtering).

        ---

        ## 🌱 5. Opportunities for Growth & Optimization
        - Identify actionable insights tied to business KPIs.
        - Use segmentation, clustering, or cross-tab analysis to discover growth potential.
        - Prioritize by impact, feasibility, and risk.

        ---

        ## 💡 6. Hidden or Surprising Insights
        - Detect any **non-obvious** trends, patterns, or behaviors.
        - Show how these findings might reveal blind spots or strategic advantages.

        ---

        ## 🧠 7. Strategic Recommendations
        - Provide **3–5 clear, data-backed actions** for decision-makers.
        - Align each with business objectives (cost savings, revenue growth, efficiency).
        - Include a "next steps" section (further data needed, A/B test, dashboard build...).

        ---

        ## 📊 Summary Table of Key Drivers

        | Category              | Factor            | Impact Level | Statistical Significance | Recommendation                      |
        |----------------------|-------------------|--------------|---------------------------|-------------------------------------|
        | 📈 High Impact       | [Variable Name]   | Strong       | ✅ p < 0.05                | [Recommended Action]               |
        | ⚠️ Low/Negative Impact | [Variable Name]   | Weak/Negative| ❌ Not significant         | [Mitigation Strategy or Ignore]    |

        ---

        ## 📌 Presentation Guidelines
        - Use professional, business-oriented language.
        - Include emojis 🎯 📈 ⚠️ 💡 💰 🔍 to enhance readability.
        - Be clear, direct, and data-driven.
        - If any part of the dataset is unclear or incomplete, ask clarifying questions before finalizing.

        Once the dataset is received, begin your advanced analysis.
        '''

        # Format the prompt with actual data values
        prompt_text = analysis_template.format(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description
        )

        try:
            # Call the large language model with the formatted prompt
            analysis_result = self.llm(prompt_text)
        except Exception as e:
            return f"Error during analysis: {str(e)}"

        # Extract the content from the LLM's response, handle different response types
        if hasattr(analysis_result, 'content'):
            analysis_content = analysis_result.content
        else:
            analysis_content = str(analysis_result)

        # Append the prompt and response to the conversation memory
        self.memory.append(HumanMessage(content=prompt_text))
        self.memory.append(AIMessage(content=analysis_content))

        # Save the analysis result in the object
        self.analysis = analysis_content

        # Optionally save analysis and prompt to the database if configured
        if hasattr(self, 'db') and self.db and hasattr(self, 'report_id') and self.report_id:
            self.db.saveMemory(
                reportID=self.report_id,
                llm=self.db.llm_id_by_name(self.llm.model),
                prompet=prompt_text,
                response=analysis_content,
                chat=False
            )

        return analysis_content
    
    def questions_gen(self, num):
        """
        Generate strategic analytical questions that can be visualized based on the data.
        
        This function uses the large language model to generate high business value questions
        that can be used to explore the data or create dashboards.
        
        Parameters:
            num (int): Number of questions to generate
            
        Returns:
            List[str]: List of generated analytical questions
            
        Notes:
            - Generated questions focus on trends, patterns, and relationships in the data.
            - Questions are stored in the conversation history (self.memory).
            - Questions are saved to the database if self.report_id is set.
            
        Example:
            >>> questions = analyzer.questions_gen(5)
            >>> for i, q in enumerate(questions, 1):
            ...     print(f"{i}. {q}")
            1. How has the sales rate evolved over the past 12 months?
            2. What are the top 5 best-selling products and what is their contribution to total sales?
            3. What is the distribution of customers by geographic region?
            4. Is there a relationship between order volume and complaint rate?
            5. Which marketing channels are most effective in terms of conversion rate?
        """
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description
        

        question_prompt = f"""
        You are a senior data analyst hired by a company to extract meaningful, high-level, and actionable business insights from the following dataset.

        Your job is to generate advanced **strategic questions** that:
        - Are deeply rooted in the data structure and semantics.
        - Reflect important **business objectives**, patterns, risks, or growth opportunities.
        - Are **strong, insightful, and relevant** to decision-makers like company owners or managers.
        - Can be **easily visualized** using bar charts, line plots, histograms, scatter plots, or pie charts.

        **DO NOT generate general or surface-level questions. Instead, focus on questions that:**
        - Quantify change over time or between groups.
        - Explore distribution, frequency, or correlation.
        - Investigate trends, seasonality, or anomalies.
        - Provide guidance for optimizing business performance or identifying risks.

        You MUST generate exactly {num} chartable, insightful questions.

        ### INPUTS:
        1. Dataset Overview: {data_info}
        2. Dataset Sample: {data_sample}
        3. Data Summary: {data_description}

        ### OUTPUT FORMAT:
        Write {num} powerful analytical questions that:
        - Could be visualized with a chart.
        - Have clear business relevance.
        - Reflect advanced reasoning.

        Each question should be written on a separate line.

        Example Questions:
        - How has the conversion rate changed over time across different marketing channels?
        - Which regions have shown the fastest growth in revenue over the past year?
        - What is the correlation between customer satisfaction scores and return frequency?
        - How does the average transaction value vary by customer segment?
        """

        question_template = PromptTemplate(
            input_variables=["num", "data_info", "data_sample", "data_description"],
            template=question_prompt
        )

        question_chain = question_template | self.llm

        try:
            generated_questions = question_chain.invoke({
                "num": num,
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description
            })

            # Ensure the response is properly encoded
            if isinstance(generated_questions, str):
                generated_questions = generated_questions.encode('utf-8', 'replace').decode('utf-8')

            print("Raw LLM Output:", repr(generated_questions))

            if not generated_questions.strip():
                print("Warning: LLM did not generate any questions.")
                return []

            # Use the improved extraction function
            questions_list = extract_questions(generated_questions)

            print("Extracted Questions List:", questions_list)

            # Trim or handle missing questions
            if len(questions_list) > num:
                questions_list = questions_list[:num]
            elif len(questions_list) < num:
                print(f"Warning: Expected {num} questions, but got {len(questions_list)}")

            # Store in memory
            formatted_question_prompt = question_template.format(
                num=num,
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            self.memory.append(HumanMessage(content=formatted_question_prompt))
            self.memory.append(AIMessage(content="\n".join(questions_list)))
            self.db.saveMemory(reportID=self.report_id,
                           llm=self.db.llm_id_by_name(self.llm.model),
                           prompet=formatted_question_prompt,
                           response="\n".join(questions_list),
                           chat=False)

            return questions_list

        except Exception as e:
            print(f"Error generating questions: {str(e)}")
            return []

    
    def chat(self, question: str) -> str:
        """
        Interact with the data analysis system to answer questions about the dataset.
        
        This function uses the large language model with previous conversation history
        to provide accurate and contextual answers about the data.
        
        Parameters:
            question (str): User's question about the data
            
        Returns:
            str: The model's response with data-informed insights
            
        Notes:
            - The question and answer are stored in the conversation history (self.memory).
            - The conversation is saved to the database if self.report_id is set.
            - The model can leverage all previous analyses in the answer.
            
        Example:
            >>> response = analyzer.chat("Which product category has the highest sales?")
            >>> print(response)
            Based on the data, the "Electronics" category has the highest sales with 37% of total sales,
            followed by "Home Appliances" with 24%.
        """
        system_prompt = f"""
        You are a data analyst with expertise in analyzing {self.dataframe.shape[1]} variables across {self.dataframe.shape[0]} records.

        Dataset context:
        - Type of data: {self.data_info.splitlines()[0] if self.data_info else 'Unknown dataset'}
        - Key columns: {', '.join(self.dataframe.columns[:5]) if len(self.dataframe.columns) > 5 else self.data_cols}

        Instructions:
        - Answer using ONLY the data available.
        - If asked about unknown variables, respond transparently.
        - Prioritize clarity, relevance, and helpfulness.
        """

        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
        ])

        chain = prompt | self.llm

        response = chain.invoke({
            "chat_history": self.memory,
            "question": question
        })

        self.memory.append(HumanMessage(content=question))
        self.memory.append(AIMessage(content=response))

        self.db.saveMemory(
            reportID=self.report_id,
            llm=self.db.llm_id_by_name(self.llm.model),
            prompet=question,
            response=response,
            chat=True
        )

        return response
    
    def select_chart_type(self, question: str) -> Dict:
        """
        Select the appropriate chart type based on the question and data with advanced visualization options.
        
        This function analyzes the question and determines the most suitable chart type
        for visualizing the answer, including advanced visualization options and configurations.
        
        Parameters:
            question (str): The analytical question to visualize
            
        Returns:
            Dict: A dictionary containing:
                - 'type': Primary chart type
                - 'subtype': Specific variant of the chart type (if applicable)
                - 'style': Visualization style/theme
                - 'color_scheme': Recommended color palette
                - 'annotations': Suggested annotations or highlights
                - 'interactivity': Recommended interactive features
            
        Notes:
            Advanced Chart Types:
            - Bar/Column: 'Bar', 'StackedBar', 'GroupedBar', 'HorizontalBar'
            - Line: 'Line', 'AreaChart', 'MultiLine', 'SparkLine'
            - Pie/Distribution: 'Pie', 'Donut', 'Treemap', 'Sunburst'
            - Scatter/Relations: 'Scatter', 'Bubble', 'HexBin', 'DensityContour'
            - Statistical: 'Histogram', 'BoxPlot', 'Violin', 'Heatmap'
            - Specialized: 'Funnel', 'Radar', 'Waterfall', 'Sankey'
            
        Example:
            >>> chart_info = analyzer.select_chart_type("How have monthly sales evolved over the past year?")
            >>> print(chart_info['type'])
            Line
            >>> print(chart_info['style'])
            Business
        """
        self.chart_type_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class data visualization expert specializing in enterprise-grade business intelligence.
            
            STEP 1: ANALYZE THE QUESTION AND DATA THOROUGHLY
            - Understand the business context and the exact analytics need
            - Identify key measurement variables and dimensions
            - Consider the audience (executives, analysts, operations)
            
            STEP 2: SELECT THE OPTIMAL CHART TYPE AND CONFIGURATION
            
            PRIMARY CHART TYPES:
            - Comparison Charts (Bar/Column): 'Bar', 'StackedBar', 'GroupedBar', 'HorizontalBar'
            - Trend Charts (Line): 'Line', 'AreaChart', 'MultiLine', 'SparkLine'
            - Composition Charts: 'Pie', 'Donut', 'Treemap', 'Sunburst'
            - Relationship Charts: 'Scatter', 'Bubble', 'HexBin', 'DensityContour'
            - Distribution Charts: 'Histogram', 'BoxPlot', 'Violin', 'Heatmap'
            - Specialized Visualizations: 'Funnel', 'Radar', 'Waterfall', 'Sankey', 'Gauge'
            
            VISUAL STYLE RECOMMENDATIONS:
            - Executive: Clean, minimalist, high contrast, focus on insights
            - Analytical: Detailed, information-rich, precise
            - Dashboard: Compact, high information density, part of a system
            - Presentation: Bold, clear messaging, storytelling elements
            
            COLOR SCHEMES:
            - Corporate: Professional blues and grays (ideal for finance, consulting)
            - Analytical: Blue to orange sequential (ideal for diverging data)
            - Impact: High contrast with strategic highlighting
            - Categorical: Distinct colors for clear differentiation
            
            ANNOTATIONS:
            - Benchmarks: Industry standards, previous periods
            - Trend indicators: Growth rates, YoY changes
            - Statistical: Mean, median, outliers
            - Contextual: Important events, milestones
            
            INTERACTIVITY FEATURES:
            - Filtering: By time period, category, region
            - Drill-down: From aggregated to detailed view
            - Tooltips: Detailed information on hover
            - Cross-filtering: Connected visualizations
            
            Data Description: {data_description}
            Available Columns: {columns}
            Sample Data: {sample_data}
            Question: {question}
            
            RESPOND ONLY WITH THE FOLLOWING JSON FORMAT:
            {{
              "type": "[PRIMARY_CHART_TYPE]",
              "subtype": "[SPECIFIC_VARIANT]",
              "style": "[VISUAL_STYLE]",
              "color_scheme": "[COLOR_PALETTE]",
              "annotations": "[RECOMMENDED_ANNOTATIONS]",
              "interactivity": "[INTERACTIVE_FEATURES]"
            }}
            """)
        ])

        self.llm.temperature = 0.3
        chain = self.chart_type_prompt | self.llm
        response = chain.invoke({
            "data_description": self.data_description,
            "columns": self.data_cols,
            "sample_data": self.data_sample,
            "question": question
        })
        self.llm.temperature = 0.7
        
        # Parse the JSON response
        try:
            # First, try to extract JSON if it's embedded in other text
            json_match = re.search(r'({[\s\S]*})', response)
            if json_match:
                response = json_match.group(1)
            
            chart_info = json.loads(response)
            
            # Validate primary chart type and provide fallback
            basic_types = {
                'Bar', 'StackedBar', 'GroupedBar', 'HorizontalBar',
                'Line', 'AreaChart', 'MultiLine', 'SparkLine',
                'Pie', 'Donut', 'Treemap', 'Sunburst',
                'Scatter', 'Bubble', 'HexBin', 'DensityContour',
                'Histogram', 'BoxPlot', 'Violin', 'Heatmap',
                'Funnel', 'Radar', 'Waterfall', 'Sankey', 'Gauge'
            }
            
            if 'type' not in chart_info or chart_info['type'] not in basic_types:
                chart_info['type'] = 'Bar'  # Default fallback
                
            # Ensure all fields exist
            required_fields = ['type', 'subtype', 'style', 'color_scheme', 'annotations', 'interactivity']
            for field in required_fields:
                if field not in chart_info:
                    chart_info[field] = ""
                    
            return chart_info
            
        except Exception as e:
            logger.error(f"Error parsing chart type response: {e}")
            # Return default values if parsing fails
            return {
                'type': 'Bar',
                'subtype': '',
                'style': 'Executive',
                'color_scheme': 'Corporate',
                'annotations': 'Benchmarks',
                'interactivity': 'Tooltips'
            }
    
    def select_columns(self, question: str) -> Dict[str, List]:
        """
        Select and organize the relevant columns based on the question for professional visualization.
        
        This function analyzes the question and determines the most appropriate columns
        to use for the analysis, including column roles and transformations.
        
        Parameters:
            question (str): The analytical question
            
        Returns:
            Dict[str, List]: Dictionary containing:
                - 'dimensions': Categorical/grouping columns
                - 'measures': Numerical/metric columns
                - 'time_dimension': Time-related columns
                - 'filters': Columns for filtering
                - 'transformations': Suggested transformations
            
        Notes:
            - Dimension columns define categories for comparison/segmentation
            - Measure columns contain the numerical values to analyze
            - Time dimensions enable trend analysis
            - Filters allow focusing on specific segments
            - Transformations suggest data operations for better visualization
            
        Example:
            >>> col_structure = analyzer.select_columns("What is the average sales per region?")
            >>> print(col_structure['dimensions'])
            ['region']
            >>> print(col_structure['measures'])
            ['sales']
            >>> print(col_structure['transformations'])
            [{'column': 'sales', 'operation': 'average'}]
        """
        self.columns_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class business intelligence architect specializing in data modeling for visualization.
            
            TASK: INTELLIGENTLY SELECT AND ORGANIZE COLUMNS FOR PROFESSIONAL DATA VISUALIZATION
            
            STEP 1: ANALYZE THE ANALYTICAL QUESTION AND DATA
            - Extract the business requirements and analytics needs
            - Identify metrics, dimensions, and criteria mentioned
            - Understand the granularity and level of detail needed
            
            STEP 2: ORGANIZE DATA INTO VISUALIZATION FRAMEWORK
            
            COLUMN TYPES TO IDENTIFY:
            1. DIMENSIONS (categorical/grouping columns):
               - Primary dimensions (main categories to compare)
               - Secondary dimensions (additional segmentation)
               - Hierarchical dimensions (drill-down paths)
            
            2. MEASURES (numerical/metric columns):
               - Primary measure (main KPI being analyzed)
               - Secondary measures (supporting metrics)
               - Calculated measures (need to be derived)
            
            3. TIME DIMENSIONS:
               - Primary time grain (e.g., month, quarter, year)
               - Temporal comparisons (e.g., YoY, MoM)
            
            4. FILTERS:
               - Columns that should be used to filter/segment data
               - Range constraints (e.g., date ranges, value thresholds)
            
            5. TRANSFORMATIONS:
               - Aggregations (sum, average, median, etc.)
               - Calculations (growth rates, percentages, ratios)
               - Formatting (number formats, categorizations)
            
            IMPORTANT RULES:
            - ONLY select columns that exist in the available columns list
            - Ensure proper data types (numeric for measures, categorical for dimensions)
            - Prioritize business relevance over technical convenience
            - Consider visualization best practices (not too many dimensions at once)
            
            Data Description: {data_description}
            Available Columns: {columns}
            Sample Data: {sample_data}
            Question: {question}
            
            RESPOND ONLY WITH THE FOLLOWING JSON FORMAT:
            {{
              "dimensions": ["dimension1", "dimension2"],
              "measures": ["measure1", "measure2"],
              "time_dimension": ["time_column"],
              "filters": ["filter_column1", "filter_column2"],
              "transformations": [
                {{"column": "column_name", "operation": "aggregation/calculation", "details": "specifics"}}
              ]
            }}
            """)
        ])

        self.llm.temperature = 0.3
        chain = self.columns_prompt | self.llm
        response = chain.invoke({
            "data_description": self.data_description,
            "columns": self.data_cols,
            "sample_data": self.data_sample,
            "question": question
        })
        self.llm.temperature = 0.7
        
        # Parse JSON response
        try:
            # First, try to extract JSON if it's embedded in other text
            json_match = re.search(r'({[\s\S]*})', response)
            if json_match:
                response = json_match.group(1)
                
            column_structure = json.loads(response)
            
            # Validate that all columns exist in the dataframe
            available_cols = self.dataframe.columns.tolist()
            
            for key in ['dimensions', 'measures', 'time_dimension', 'filters']:
                if key in column_structure:
                    column_structure[key] = [col for col in column_structure[key] if col in available_cols]
            
            # Ensure all required keys exist
            required_keys = ['dimensions', 'measures', 'time_dimension', 'filters', 'transformations']
            for key in required_keys:
                if key not in column_structure:
                    column_structure[key] = []
            
            # Validate transformations
            if 'transformations' in column_structure and isinstance(column_structure['transformations'], list):
                valid_transformations = []
                for transform in column_structure['transformations']:
                    if isinstance(transform, dict) and 'column' in transform and transform['column'] in available_cols:
                        valid_transformations.append(transform)
                column_structure['transformations'] = valid_transformations
            
            return column_structure
            
        except Exception as e:
            logger.error(f"Error parsing column structure response: {e}")
            # Return a basic structure if parsing fails
            all_columns = self.dataframe.columns.tolist()
            numeric_cols = self.dataframe.select_dtypes(include=np.number).columns.tolist()
            categorical_cols = list(set(all_columns) - set(numeric_cols))
            
            # Simple fallback based on data types
            return {
                'dimensions': categorical_cols[:2] if categorical_cols else [],
                'measures': numeric_cols[:2] if numeric_cols else [],
                'time_dimension': [],
                'filters': [],
                'transformations': []
            }
    
    def get_chart_recommendation(self, question: str) -> Dict:
        """
        Get a comprehensive chart recommendation based on a question for professional visualization.
        
        This function provides a complete visualization recommendation including chart type,
        data selection, styling, annotations, and interactivity options suitable for
        professional business presentations.
        
        Parameters:
            question (str): The analytical question to visualize
            
        Returns:
            Dict: A comprehensive visualization configuration with these keys:
                - 'chart': Chart type and configuration
                - 'data': Data columns and structure
                - 'title': Suggested title for the visualization
                - 'subtitle': Suggested subtitle with key insight
                - 'styling': Visual styling recommendations
                - 'annotations': Recommended annotations
                - 'interactivity': Suggested interactive features
                - 'alternative_views': Alternative visualization options
                
        Notes:
            - The recommendation is tailored for professional business contexts
            - Focuses on clear communication of insights
            - Includes executive-ready styling and annotations
            - Provides interactivity options for deeper exploration
            
        Example:
            >>> recommendation = analyzer.get_chart_recommendation(
            ...     "What are the top 5 best-selling products by revenue?"
            ... )
            >>> print(recommendation['chart']['type'])
            HorizontalBar
            >>> print(recommendation['title'])
            "Top 5 Products by Revenue"
        """
        # Get detailed chart type and column structure
        chart_info = self.select_chart_type(question)
        column_structure = self.select_columns(question)
        
        # Now generate a complete visualization recommendation
        self.recommendation_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class business intelligence visualization expert.
            
            TASK: CREATE AN EXECUTIVE-GRADE VISUALIZATION RECOMMENDATION
            
            I'll provide you with:
            1. An analytical question
            2. Recommended chart type/configuration 
            3. Recommended data structure
            
            Your task is to create a COMPREHENSIVE, PROFESSIONAL visualization recommendation
            that would impress senior executives at leading global firms.
            
            Question: {question}
            
            Chart Information: {chart_info}
            
            Data Structure: {column_structure}
            
            Sample Data: {sample_data}
            
            DEVELOP A COMPLETE VISUALIZATION RECOMMENDATION INCLUDING:
            
            1. EFFECTIVE TITLE & SUBTITLE:
               - Clear, concise title communicating the main point
               - Insightful subtitle highlighting key finding/trend
            
            2. PROFESSIONAL STYLING:
               - Color palette recommendations (exact hex codes)
               - Typography suggestions (font family, sizes)
               - Layout recommendations (proportions, whitespace)
               - Gridlines, axes, and legend configuration
            
            3. STRATEGIC ANNOTATIONS:
               - Key callouts to highlight important insights
               - Trend indicators (arrows, growth rates)
               - Benchmark/reference lines or regions
               - Contextual notes where appropriate
            
            4. INTERACTIVE ELEMENTS:
               - Filters and slicers for exploration
               - Tooltips with rich information
               - Drill-down capabilities
               - Cross-filtering connections
            
            5. ALTERNATIVE VISUALIZATION OPTIONS:
               - 1-2 alternative approaches with pros/cons
            
            Follow these visualization best practices:
            - Minimize chart junk and maximize data-ink ratio
            - Use color strategically to highlight insights
            - Ensure accessibility (color-blind friendly)
            - Design for the intended audience (executives)
            
            RESPOND ONLY WITH THE FOLLOWING JSON FORMAT:
            {{
              "chart": {{
                "type": "{chart_info['type']}",
                "subtype": "{chart_info['subtype']}",
                "orientation": "vertical/horizontal",
                "stacking": "normal/percent/none"
              }},
              "data": {{
                "dimensions": {column_structure['dimensions']},
                "measures": {column_structure['measures']},
                "time_dimension": {column_structure['time_dimension']},
                "filters": {column_structure['filters']},
                "transformations": {column_structure['transformations']},
                "sort_by": "measure_name",
                "sort_order": "ascending/descending",
                "limit": "top/bottom N"
              }},
              "title": "Suggested title",
              "subtitle": "Insightful subtitle with key finding",
              "styling": {{
                "color_palette": ["#hexcode1", "#hexcode2"],
                "background": "#hexcode",
                "font_family": "font name",
                "grid_lines": "minimal/none/full",
                "theme": "professional theme name"
              }},
              "annotations": [
                {{"type": "highlight", "element": "specific element", "reason": "why highlight"}},
                {{"type": "reference_line", "value": "value/metric", "label": "label text"}}
              ],
              "interactivity": [
                {{"feature": "feature name", "description": "how it enhances analysis"}}
              ],
              "alternative_views": [
                {{"chart_type": "alternative type", "advantage": "specific benefit"}}
              ]
            }}
            """)
        ])
        
        self.llm.temperature = 0.4
        chain = self.recommendation_prompt | self.llm
        response = chain.invoke({
            "question": question,
            "chart_info": chart_info,
            "column_structure": column_structure,
            "sample_data": self.data_sample
        })
        self.llm.temperature = 0.7
        
        # Parse the response
        try:
            # Extract JSON if embedded in text
            json_match = re.search(r'({[\s\S]*})', response)
            if json_match:
                response = json_match.group(1)
                
            recommendation = json.loads(response)
            
            # Apply some additional metadata
            recommendation['generated_for'] = {
                'question': question,
                'timestamp': datetime.datetime.now().isoformat(),
                'dataset_rows': len(self.dataframe),
                'dataset_columns': len(self.dataframe.columns)
            }
            
            return recommendation
            
        except Exception as e:
            logger.error(f"Error parsing visualization recommendation: {e}")
            # Create a fallback recommendation
            chart_type = chart_info.get('type', 'Bar')
            
            dimensions = column_structure.get('dimensions', [])
            measures = column_structure.get('measures', [])
            
            return {
                'chart': {
                    'type': chart_type,
                    'subtype': chart_info.get('subtype', ''),
                    'orientation': 'vertical',
                    'stacking': 'none'
                },
                'data': {
                    'dimensions': dimensions,
                    'measures': measures,
                    'time_dimension': column_structure.get('time_dimension', []),
                    'filters': column_structure.get('filters', []),
                    'transformations': column_structure.get('transformations', []),
                    'sort_by': measures[0] if measures else '',
                    'sort_order': 'descending',
                    'limit': 'top 10'
                },
                'title': f"Analysis of {', '.join(measures)} by {', '.join(dimensions)}" if measures and dimensions else "Data Analysis",
                'subtitle': "Key insights visualization",
                'styling': {
                    'color_palette': ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"],
                    'background': "#ffffff",
                    'font_family': "Arial",
                    'grid_lines': "minimal",
                    'theme': "professional"
                },
                'annotations': [],
                'interactivity': [{"feature": "tooltip", "description": "Show details on hover"}],
                'alternative_views': [],
                'generated_for': {
                    'question': question,
                    'timestamp': datetime.datetime.now().isoformat(),
                    'dataset_rows': len(self.dataframe),
                    'dataset_columns': len(self.dataframe.columns)
                }
            }
    
    def generate_recommendations(self, num_recommendations: int = 5):
        """
        Generate strategic business recommendations based on data analysis.
        
        This function uses the large language model with previous data analysis to create
        actionable recommendations with business impact.
        
        Parameters:
            num_recommendations (int, optional): Number of recommendations to generate. Default is 5.
            
        Returns:
            str: Text containing recommendations organized in a table and with full details
            
        Notes:
            - Recommendations are based on analysis results (self.analysis).
            - Recommendations include title, details, expected impact, and potential risks.
            - Risks are categorized with emojis: ✅ (low), ⚠️ (medium), ❗(high).
            - Recommendations are stored in the conversation history (self.memory).
            - Recommendations are saved to the database if self.report_id is set.
            
        Example:
            >>> recommendations = analyzer.generate_recommendations(3)
            >>> print(recommendations[:500])  # Display just the beginning of the recommendations
        """
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description
        analysis = self.analysis  # التحليل الذي تم عمله سابقاً

        recommendation_prompt = '''
        You are a world-class business consultant and data analyst.

        You have analyzed the following:
        - Dataset metadata: {data_info}
        - Dataset sample: {data_sample}
        - Dataset summary: {data_description}
        - Detailed business analysis: {analysis}

        Based on your deep understanding of the data and analysis:
        Your task is to generate {num_recommendations} highly actionable, strategic recommendations for the business.

        Your recommendations must:
        - Be directly based on the analysis and insights.
        - Address clear business actions (e.g., optimize processes, launch new products, reduce risks, target specific segments, etc.)
        - Be specific, impactful, and feasible.
        - Cover both short-term quick wins and long-term strategic moves.
        - Include estimated expected outcome in percentage (%) where appropriate.
        - Include any potential risks or challenges for each recommendation.
        - Reference relevant metrics or insights from the analysis if possible.
        - Use professional, executive-level language.
        - Add an appropriate emoji based on risk level:
            - ✅ for Low risk
            - ⚠️ for Medium risk
            - ❗for High risk

        Output Format:

        ### 📋 Recommendations Table

        | # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
        |---|-----------------------|---------------------|-----------------------------|
        | 1 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | 2 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | ... | ... | ... | ... |

        ---

        ### 📋 Full Recommendation Details

        1. **[Recommendation Title]** [Emoji]
        - **Details:** Explain clearly what should be done and why.
        - **Expected Impact:** [e.g., Increase attendance by 10%]
        - **Metrics Reference:** [Reference specific metric if available, e.g., matches with <50% attendance]
        - **Potential Risks:** [Possible challenges or risks involved]
        - **Timeline:** [Short-term or Long-term]

        Repeat similarly for each recommendation.
        '''

        
        rec_template = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "analysis", "num_recommendations"],
            template=recommendation_prompt
        )

        rec_chain = LLMChain(llm=self.llm, prompt=rec_template)

        rec_response = rec_chain.run(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            num_recommendations=num_recommendations
        )

        formatted_rec_prompt = recommendation_prompt.format(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            num_recommendations=num_recommendations
        )
        self.memory.append(HumanMessage(content=formatted_rec_prompt))
        self.memory.append(AIMessage(content=rec_response))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_rec_prompt,
                        response=rec_response,
                        chat=False)

        return rec_response 

In [7]:
import os
import pandas as pd
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from DataAnalyzer import DataAnalyzer
import pandas as pd
# Load environment variables
load_dotenv()

# Initialize the LLM
groq_api_key = os.environ['GROQ_API_KEY']
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama3-8b-8192"
)

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.analysis_data()

# Print the analysis
print(analysis_result)


ValidationError: 21 validation errors for AIMessage
content.str
  Input should be a valid string [type=string_type, input_value=AIMessage(content="**Exec..., 'total_tokens': 3381}), input_type=AIMessage]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type
content.list[union[str,dict[any,any]]].0.str
  Input should be a valid string [type=string_type, input_value=('content', "**Executive ...s and recommendations."), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type
content.list[union[str,dict[any,any]]].0.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=('content', "**Executive ...s and recommendations."), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.10/v/dict_type
content.list[union[str,dict[any,any]]].1.str
  Input should be a valid string [type=string_type, input_value=('additional_kwargs', {}), input_type=tuple]
    For further information